# מסגרת סוכן מיקרוסופט — Azure OpenAI (ממשק API לתגובות)

בדוגמת קוד זו, תשתמש ב**מסגרת הסוכן של מיקרוסופט (MAF)** כדי ליצור סוכן פשוט המגובה על ידי **Azure OpenAI** באמצעות **ממשק API לתגובות**.

> **הערת הגירה:** דוגמה זו השתמשה בעבר ב-Semantic Kernel עם מודלים של GitHub. היא הועלתה למסגרת הסוכן של מיקרוסופט, ומודלים של GitHub (שהם מיושנים ומסוננים עד יולי 2026) הוחלפו ב-Azure OpenAI, התומכת בממשק ה-API לתגובות. ה-`OpenAIChatClient` במסגרת MAF מיועד לכתובת היציבה `/openai/v1/` של Azure OpenAI ומשתמש כברירת מחדל ב-API לתגובות.

מטרת הדוגמה הזו היא להציג את השלבים שיוצגו לאחר מכן בדוגמאות קוד נוספות בעת יישום דפוסי סוכן שונים.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## ייבא את חבילות הפייתון הנדרשות


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## הגדרת כלי

במסגרת Microsoft Agent Framework, **כלי** הוא פונקציית פייתון פשוטה המעוטרת ב-`@tool` שהסוכן יכול לקרוא לה. למטה אנו מגדירים כלי שמחזיר יעד חופשה אקראי ומונע חזרה על היעד הקודם.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## יצירת הסוכן

כאן, אנו יוצרים את הסוכן בשם `TravelAgent`.

בדוגמה זו, אנו משתמשים בהוראות בסיסיות מאוד. אתה מוזמן לשנות הוראות אלו כדי להבחין כיצד מתנהג הסוכן משתנה.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## הפעלת הסוכן

עכשיו ניתן להפעיל את הסוכן. אנו יוצרים `AgentSession` כדי שהסוכן יזכור את השיחה לאורך הסיבובים, ואז שולחים שני `user_inputs`. הראשון מבקש טיול; השני אומר שהמשתמש לא אהב את ההצעה ומבקש עוד אחת — הסוכן משתמש בהיסטוריית הסשן ובכלי `get_random_destination` כדי להגיב.

ניתן לשנות את ההודעות האלו כדי לצפות איך הסוכן מגיב שונה. התגובות **מוזרמות** אסימן-אסימן.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
